In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py
import pandas as pd
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100000)
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
import pymysql
from datetime import datetime,timedelta
import matplotlib.pyplot as plt

analytics_mongo = client_db.get_database('analytics_db')



db2 = client2.get_database('telematics_db')



latitude=[8,38]
longitude=[68,98]
bottomLeft = (8.0, 68.0)
bottomRight = (8.0, 98.0)
topLeft = (38.0, 68.0)
topRight = (38.0, 98.0)

cols1 = np.linspace(bottomLeft[1], bottomRight[1], num=288)
rows1 = np.linspace(bottomLeft[0], topLeft[0], num=335)


count=1
row=1
grid_rows=[]
for i in rows1:
    col=1
    for j in cols1:
        grid_rows.append([count,row,col,i,i+30/335,j,j+30/288])
        count+=1
        col+=1
    row+=1
GRID_TABLE=pd.DataFrame(grid_rows,columns=['layer1','layer1_row','layer1_col','layer1_from_lat','layer1_to_lat','layer1_from_lon','layer1_to_lon'])    
GRID_TABLE['layer1_centre_lat']=(GRID_TABLE['layer1_from_lat']+GRID_TABLE['layer1_to_lat'])/2
GRID_TABLE['layer1_centre_lon']=(GRID_TABLE['layer1_from_lon']+GRID_TABLE['layer1_to_lon'])/2




GRID_TABLE['layer2_row']=np.searchsorted([i for i in range (0,335,5)],GRID_TABLE['layer1_row'] )
GRID_TABLE['layer2_col']=np.searchsorted([i for i in range (0,291,5)],GRID_TABLE['layer1_col'] )
GRID_TABLE['layer2']=(GRID_TABLE['layer2_row']-1)*58+GRID_TABLE['layer2_col']


layer_2_from_to=[]
for i in GRID_TABLE.layer2.unique():
    a=GRID_TABLE[GRID_TABLE['layer2']==i]
    #a.sort_values('layer1',ascending=True,inplace=True)
    [from_lat,from_lon]=a[a['layer1']==a.layer1.min()][['layer1_from_lat','layer1_from_lon']].values[0]
    [to_lat,to_lon]=a[a['layer1']==a.layer1.max()][['layer1_to_lat','layer1_to_lon']].values[0]
    centre_lat=(from_lat+to_lat)/2
    centre_lon=(from_lon+to_lon)/2
    layer_2_from_to.append([i,from_lat,from_lon,to_lat,to_lon,centre_lat,centre_lon])

layer_2_details=pd.DataFrame(layer_2_from_to,columns=['layer2','layer2_from_lat','layer2_from_lon','layer2_to_lat','layer2_to_lon','layer2_centre_lat','layer2_centre_lon'])    

GRID_TABLE=pd.merge(GRID_TABLE,layer_2_details,on='layer2',how='left')


GRID_TABLE['layer3_row']=np.searchsorted([i for i in range (0,68,2)],GRID_TABLE['layer2_row'] )
GRID_TABLE['layer3_col']=np.searchsorted([i for i in range (0,59,2)],GRID_TABLE['layer2_col'] )
GRID_TABLE['layer3']=(GRID_TABLE['layer3_row']-1)*29+GRID_TABLE['layer3_col']


layer_3_from_to=[]
for i in GRID_TABLE.layer3.unique():
    a=GRID_TABLE[GRID_TABLE['layer3']==i]
    [from_lat,from_lon]=a[a['layer1']==a.layer1.min()][['layer1_from_lat','layer1_from_lon']].values[0]
    [to_lat,to_lon]=a[a['layer1']==a.layer1.max()][['layer1_to_lat','layer1_to_lon']].values[0]
    centre_lat=(from_lat+to_lat)/2
    centre_lon=(from_lon+to_lon)/2
    layer_3_from_to.append([i,from_lat,from_lon,to_lat,to_lon,centre_lat,centre_lon])

layer_3_details=pd.DataFrame(layer_3_from_to,columns=['layer3','layer3_from_lat','layer3_from_lon','layer3_to_lat','layer3_to_lon','layer3_centre_lat','layer3_centre_lon'])     

GRID_TABLE=pd.merge(GRID_TABLE,layer_3_details,on='layer3',how='left')


GRID_TABLE.to_csv("INDIA_GRID_TABLE.csv")

In [ ]:
GRID_TABLE=pd.read_csv("INDIA_GRID_TABLE.csv")
GRID_TABLE.layer1=GRID_TABLE.layer1.apply(int)

In [ ]:
GRID_TABLE.to_sql("india_grid_table")

In [ ]:
def centre_reverse_geocode(values):
    try:
        query={"location": {"$geoWithin": { "$centerSphere": [ [values[1],values[2]], 1/1000 ]}}}
        a=pd.DataFrame(list(db.reverseGeocode.find(query)))
    except Exception as e:
        print("exception one")
        print(e)
        a=pd.DataFrame()
    if len(a)>0:
        try:
            try:
                data=[values[0]]+a[['city','district','state','postalCode']].values[0].tolist()
                print(data)
                return data
            except:
                data=[values[0]]+[None]+a[['district','state','postalCode']].values[0].tolist()
                print(data)
                return data

        except Exception as e:
            print("exception two")
            print(e)
            return [values[0]]+[None]*5
    else:
        print("no_vlaue")
        return [values[0]]+[None]*5
    
    


In [ ]:
import multiprocessing
pool = multiprocessing.Pool()
results=pool.map()
pool.close()
pool.join()

In [ ]:
results=[i[0:5] if len(i)==6 else i for i in results ]


In [ ]:
layer3_geo=pd.DataFrame(results,columns=['layer3','layer3_city','layer3_district','layer3_state','layer3_postalCode'])

def lat_lon_layer(lat,lon):
    global GRID_TABLE
    a=GRID_TABLE[GRID_TABLE['layer1_from_lat']<lat][GRID_TABLE['layer1_to_lat']>lat][GRID_TABLE['layer1_to_lon']>lon][GRID_TABLE['layer1_from_lon']<lon]                    
    return a
    
    

In [ ]:
GRID_TABLE_trial=pd.merge(GRID_TABLE_trial,layer3_geo,on='layer3',how='left')

In [ ]:
GRID_TABLE_trial.to_csv("grid_table_with_locations_trial.csv")

In [ ]:
GRID_TABLE_trial.layer3.nunique()

In [ ]:
GRID_TABLE_trial.to_csv("INDIA_GRID_TABLE.csv",index=False)


In [ ]:
del GRID_TABLE_trial['Unnamed: 0']

In [ ]:
GRID_TABLE_trial.layer1_state.str.upper().unique()

In [ ]:
GRID_TABLE_trial.replace({'उत्तर प्रदेश':'Uttar Pradesh','हरियाणा':'HARYANA','मध्य प्रदेश':'MADHYA PRADESH','Jammu and Kashmir':'JAMMU & KASHMIR'}).layer1_state.nunique()

In [ ]:
GRID_TABLE_trial.to_sql("india_grid_table",con=engine,if_exists='append',chunksize=500,method='multi')

In [ ]:
GRID_TABLE=pd.read_sql("select * from india_grid_table",engine)

In [ ]:
GRID_TABLE.layer1_district.isna().value_counts()